In [3]:
import glob, os, re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def chunk_file(path, min_words=60, max_words=200):
    lines = [l.strip() for l in open(path, encoding="utf-8", errors="ignore").read().split("\n") if l.strip()]
    chunks, cur = [], []
    for line in lines:
        cur.append(line)
        if len(" ".join(cur).split()) >= min_words:
            chunks.append(" ".join(cur)); cur = []
    if cur and len(" ".join(cur).split()) >= 20:
        chunks.append(" ".join(cur))
    out = []
    for c in chunks:
        w = c.split()
        for i in range(0, len(w), max_words):
            out.append(" ".join(w[i:i + max_words]))
    return out

docs = []
for f in sorted(glob.glob("data/raw/protocols/*.txt")):
    for n, c in enumerate(chunk_file(f)):
        docs.append({"id": f"{os.path.basename(f)}#passage-{n}", "text": c})
kb = pd.DataFrame(docs)
print(len(kb), "passages from", len(glob.glob("data/raw/protocols/*.txt")), "files")
print(kb["id"].str.split("#").str[0].value_counts())

vec_kb = TfidfVectorizer(ngram_range=(1, 2), stop_words="english", sublinear_tf=True)
M = vec_kb.fit_transform(kb["text"])

COVERED = {"flood"}   # hazard types our saved documents actually cover
QUERY_BY_TYPE = {"flood": "flooding street sewer catch basin stormwater what residents should do during a storm"}

def attach_guidance(hazard_type, text="", min_score=0.12):
    if hazard_type not in COVERED:
        return None                                  # no documents for this hazard
    q = QUERY_BY_TYPE[hazard_type] + " " + text
    s = cosine_similarity(vec_kb.transform([q]), M)[0]
    i = int(s.argmax())
    if s[i] < min_score:
        return None
    return {"guidance_source": kb.iloc[i]["id"], "score": round(float(s[i]), 3)}

315 passages from 7 files
id
hmp_flooding.txt                    235
nycem_flooding.txt                   42
dep_plan_for_intense_storms.txt      12
dep_prepare_for_the_storm.txt        10
dep_monitor_during_the_storm.txt      7
dep_intro.txt                         5
dep_recover_after_the_storm.txt       4
Name: count, dtype: int64


In [4]:
relevant = ["sewer backup in the street", "catch basin clogged flooding", "street flooding cars stuck",
            "basement flooded with water", "flash flood warning", "what to do during a flood",
            "flood water on the road", "storm drain overflow"]
irrelevant = ["my neighbor plays loud music", "nice weather today", "traffic signal timing broken",
              "restaurant food poisoning", "illegal parking on my block", "tree fell on the road",
              "power is out in my building", "garbage not collected"]

def top_score(q):
    s = cosine_similarity(vec_kb.transform([q]), M)[0]
    return round(float(s.max()), 3)

print("relevant:  ", [top_score(q) for q in relevant])
print("irrelevant:", [top_score(q) for q in irrelevant])

relevant:   [0.097, 0.221, 0.149, 0.165, 0.316, 0.109, 0.115, 0.131]
irrelevant: [0.109, 0.116, 0.062, 0.167, 0.092, 0.11, 0.184, 0.111]


In [5]:
for q in relevant:
    s = cosine_similarity(vec_kb.transform([q]), M)[0]
    i = int(s.argmax())
    print(f"\n>> {q}\n   score {s[i]:.3f} | {kb.iloc[i]['id']}\n   {kb.iloc[i]['text'][:350]}...")


>> sewer backup in the street
   score 0.097 | dep_prepare_for_the_storm.txt#passage-5
   But we all have a role to play: If you rent, own, or live in a 1–4 family home, you should: Clear debris from the curb line and on top of catch basins so stormwater can flow freely down the street and into the sewer system. Set up sandbags, deployable barriers, and other tools to protect your property when intense storms are expected....

>> catch basin clogged flooding
   score 0.221 | nycem_flooding.txt#passage-39
   The City has several plans that focus on how to prepare for, respond to, and recover from flooding events. Before, during, and after a flooding event, the City monitors weather forecasts, and focuses on strategies such as catch basin cleaning and maintenance, and providing recovery assistance to affected households. More Resources Preparedness for ...

>> street flooding cars stuck
   score 0.149 | hmp_flooding.txt#passage-174
   New York City’s pumping stations convey millions of 

## Findings

- Retrieval baseline: TF-IDF over 315 passages from 7 public NYC pages (HMP flooding profile 235 passages, NYCEM flooding 42, four DEP Rainfall Ready pages 38 in total); the HMP page supplies 75% of the index.
- Hand-scored top-1 passage for 8 flood queries (one reviewer): 3 yes, 4 partly, 1 no. Typical failures: wrong phase (before vs during/after a flood) and generic infrastructure text.
- Scores do not separate relevant from off-topic queries (relevant 0.097–0.316, off-topic 0.062–0.184). Cutoff 0.12 keeps 5 of 8 relevant and lets 2 of 8 off-topic through, so retrieval is gated by hazard type first.
- Coverage: flooding only. Tree and power incidents get no guidance (the saved pages do not cover them).
- The pages are mitigation and public guidance, not operational dispatch procedures.
- The incident node stores a source id and score, not copied text.
- Next: sentence embeddings, passages that respect headings, documents on trees and power outages.

In [12]:
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer("all-MiniLM-L6-v2")
KB_E = emb.encode(list(kb["text"]), normalize_embeddings=True, batch_size=32)

def top_emb(q, k=1):
    s = KB_E @ emb.encode([q], normalize_embeddings=True)[0]
    return [(kb.iloc[j]["id"], round(float(s[j]), 3)) for j in s.argsort()[::-1][:k]]

print("relevant:  ", [top_emb(q)[0][1] for q in relevant])
print("irrelevant:", [top_emb(q)[0][1] for q in irrelevant])

for q in relevant:
    pid, sc = top_emb(q)[0]
    print(f"\n>> {q}\n   score {sc} | {pid}\n   {kb.loc[kb['id'] == pid, 'text'].iloc[0][:350]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

relevant:   [0.567, 0.553, 0.569, 0.69, 0.757, 0.656, 0.591, 0.467]
irrelevant: [0.264, 0.306, 0.159, 0.388, 0.347, 0.42, 0.529, 0.304]

>> sewer backup in the street
   score 0.567 | dep_plan_for_intense_storms.txt#passage-4
   The Department of Design and Construction (DDC) and DEP will complete three high level storm sewer projects, six traditional sewer projects, and install 1,300 more green infrastructure assets by June 2023. These projects will complement the existing sewer system and help manage stormwater in Brooklyn, Queens, and Staten Island. But we all have a ro...

>> catch basin clogged flooding
   score 0.553 | hmp_flooding.txt#passage-156
   NYC Local Law 48 of 2015 NYC Local Law 48 of 2015 (Law No. 2015/048) mandated that catch basins — storm drains or sewer grates that collect storm water — be cleaned and maintained annually instead of the previously mandated three-year cycle. If a nonfunctioning catch basin is found, it must be fixed within nine days. NYC Local Law 17

## Findings: embedding retrieval (all-MiniLM-L6-v2)

- Same 315 passages and same 8 flood queries as the TF-IDF test. Hand-scored top-1 passage (one reviewer, from 350-character previews): embeddings 5 yes / 3 partly / 0 no, versus TF-IDF 3 yes / 4 partly / 1 no.
- Scores separate better: relevant queries 0.467–0.757, off-topic 0.159–0.529 (only "power is out in my building" scored high, 0.529). A cutoff of ~0.50 keeps 7 of 8 relevant and passes 1 of 8 off-topic; tuned on the same 16 queries, so optimistic.
- Remaining limit: phase mismatch ("what to do during a flood" returned after-flood cleanup guidance). Coverage is still flood only.
- Small query set written by the author; single reviewer.